# 4.6 Obstacle Investigation

Per-tile 2-D map of the labeled point cloud with all extracted clusters overlaid.  
**Click any cluster dot** on the map → top-view and side-view appear on the right.

Use the **Tile** dropdown to switch between tiles.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve()))

import io
import base64
import numpy as np
import pandas as pd
import laspy

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import ipywidgets as widgets
from IPython.display import display
import plotly.graph_objects as go
from PIL import Image as PILImage

from config import CLUSTERS_DIR, LABELED_DIR

In [ ]:
inv = pd.read_csv(CLUSTERS_DIR / 'inventory.csv')
TILECODES = sorted(inv['tilecode'].unique().tolist())
print(f'Loaded {len(inv)} clusters across {len(TILECODES)} tile(s)')
print(inv.groupby('label')['cluster_idx'].count().rename('count').to_string())

In [ ]:
LABEL_NAMES = {
    0:  'Unknown',
    1:  'Road',
    9:  'Ground',
    10: 'Building',
    30: 'Tree',
    40: 'Car',
    60: 'Street Light',
    83: 'Large Container',
}

# Cluster centroid dot colours (plotly + matplotlib)
DOT_COLORS = {
    0:  '#aaaaaa',
    30: '#44ee44',
    40: '#ff8800',
    60: '#44aaff',
    83: '#dd44dd',
}
DOT_DEFAULT = '#ffffff'

# Background label → RGB float for the 2-D grid image
LABEL_PRIORITY = {10: 8, 1: 7, 30: 6, 40: 5, 60: 4, 83: 4, 79: 3, 90: 3, 9: 2, 0: 1}
BG_RGB = {
    -1: (0.07, 0.07, 0.07),
     0: (0.22, 0.22, 0.22),
     1: (0.75, 0.20, 0.20),
     9: (0.50, 0.50, 0.50),
    10: (0.20, 0.35, 0.70),
    30: (0.20, 0.65, 0.20),
    40: (1.00, 0.50, 0.10),
    60: (1.00, 0.95, 0.20),
    79: (0.80, 0.40, 0.00),
    83: (0.70, 0.20, 0.70),
    90: (0.90, 0.60, 0.10),
}
GRID_RES = 0.25  # metres per pixel

In [ ]:
# ── tile loading + background image ───────────────────────────────────────────
_tile_cache = {}   # tilecode -> (xy, labels)
_bg_cache   = {}   # tilecode -> (b64_png_str, [x_min, x_max, y_min, y_max])


def _load_tile_raw(tilecode):
    if tilecode in _tile_cache:
        return _tile_cache[tilecode]
    laz_path = LABELED_DIR / f'bgt_labeled_{tilecode}.laz'
    print(f'  Loading {laz_path.name}…', end=' ', flush=True)
    pc  = laspy.read(laz_path)
    xy  = np.column_stack([np.asarray(pc.x, np.float32),
                           np.asarray(pc.y, np.float32)])
    has = 'label' in pc.point_format.extra_dimension_names
    lbl = np.asarray(pc.label, np.int32) if has else np.zeros(len(xy), np.int32)
    print(f'{len(xy):,} pts')
    _tile_cache[tilecode] = (xy, lbl)
    return xy, lbl


def _make_bg_image(xy, labels):
    x, y    = xy[:, 0], xy[:, 1]
    x_min   = float(x.min());  y_min = float(y.min())
    xi = np.floor((x - x_min) / GRID_RES).astype(np.int32)
    yi = np.floor((y - y_min) / GRID_RES).astype(np.int32)
    nx, ny  = int(xi.max()) + 1, int(yi.max()) + 1
    prio    = np.vectorize(lambda l: LABEL_PRIORITY.get(int(l), 0))(labels)
    order   = np.argsort(prio)          # ascending → highest prio written last
    grid    = np.full((ny, nx), -1, np.int32)
    grid[yi[order], xi[order]] = labels[order]
    rgb     = np.full((ny, nx, 3), BG_RGB[-1], np.float32)
    for lv, c in BG_RGB.items():
        mask = grid == lv
        if mask.any():
            rgb[mask] = c
    extent  = [x_min, x_min + nx * GRID_RES, y_min, y_min + ny * GRID_RES]
    return rgb, extent


def _rgb_to_b64(rgb):
    """HxWx3 float array → base64 PNG data-URL (y-flipped for plotly)."""
    img_u8  = (rgb[::-1] * 255).clip(0, 255).astype(np.uint8)
    pil     = PILImage.fromarray(img_u8, 'RGB')
    buf     = io.BytesIO()
    pil.save(buf, format='PNG')
    return 'data:image/png;base64,' + base64.b64encode(buf.getvalue()).decode()


def _get_tile_bg(tilecode):
    if tilecode not in _bg_cache:
        xy, lbl = _load_tile_raw(tilecode)
        rgb, ext = _make_bg_image(xy, lbl)
        _bg_cache[tilecode] = (_rgb_to_b64(rgb), ext)
    return _bg_cache[tilecode]

In [ ]:
# ── detail view (top + side), rendered to PNG via Agg ─────────────────────────
def _sa(ax):
    ax.set_facecolor('#1a1a1a')
    for sp in ax.spines.values():
        sp.set_edgecolor('#444')
    ax.tick_params(labelsize=6, colors='grey')


def _fig_to_png(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=110,
                facecolor='#1a1a1a', bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    return buf.read()


def render_detail(row):
    """Return PNG bytes showing top-view (XY) and side-view (XZ) for a cluster."""
    try:
        npz = np.load(row['npz_path'])
    except Exception as e:
        fig, ax = plt.subplots(figsize=(9, 4))
        fig.patch.set_facecolor('#1a1a1a')
        _sa(ax)
        ax.text(0.5, 0.5, f'Could not load NPZ:\n{e}',
                color='#cc4444', ha='center', va='center',
                transform=ax.transAxes, fontsize=9)
        return _fig_to_png(fig)

    xyz = npz['xyz_centered']           # (N, 3)  XY centred
    hag = npz.get('height_ag', None)
    if hag is None or np.all(np.isnan(hag)):
        hag = xyz[:, 2] - xyz[:, 2].min()
    hag    = np.nan_to_num(hag, nan=0.0)
    colors = plt.cm.plasma(np.clip(hag, 0, 6) / 6.0)
    pt_sz  = max(1, min(10, 3000 // max(len(xyz), 1)))

    lbl  = int(row['label'])
    name = LABEL_NAMES.get(lbl, f'Label {lbl}')
    suptitle = (
        f"#{int(row['cluster_idx'])}  {name}  ·  "
        f"{int(row['n_raw_pts']):,} pts  ·  "
        f"{float(row['area_m2']):.2f} m²  ·  "
        f"{row.get('label_source', '')}"
    )

    fig, (ax_t, ax_s) = plt.subplots(1, 2, figsize=(10, 4.5))
    fig.patch.set_facecolor('#1a1a1a')
    fig.suptitle(suptitle, color='white', fontsize=8)
    _sa(ax_t); _sa(ax_s)

    # top view – XY
    ax_t.scatter(xyz[:, 0], xyz[:, 1], c=colors, s=pt_sz, linewidths=0)
    ax_t.set_aspect('equal')
    ax_t.set_title('top view (XY)', color='#aaaaaa', fontsize=8)
    ax_t.set_xlabel('ΔX (m)', color='grey', fontsize=7)
    ax_t.set_ylabel('ΔY (m)', color='grey', fontsize=7)

    # side view – XZ
    ax_s.scatter(xyz[:, 0], xyz[:, 2], c=colors, s=pt_sz, linewidths=0)
    ax_s.set_aspect('equal')
    ax_s.set_title('side view (XZ)', color='#aaaaaa', fontsize=8)
    ax_s.set_xlabel('ΔX (m)', color='grey', fontsize=7)
    ax_s.set_ylabel('height (m)', color='grey', fontsize=7)

    plt.tight_layout()
    return _fig_to_png(fig)

In [ ]:
# ── widgets ────────────────────────────────────────────────────────────────────
tile_dd = widgets.Dropdown(
    options=TILECODES, value=TILECODES[0],
    description='Tile:',
    layout=widgets.Layout(width='320px'),
    style={'description_width': '40px'},
)

detail_img = widgets.Image(
    value=b'', format='png',
    layout=widgets.Layout(width='540px'),
)

info_html = widgets.HTML(
    value='<span style="color:#666;font-size:12px">click a cluster dot on the map</span>',
)

# plotly FigureWidget — clickable map
fig_map = go.FigureWidget()
fig_map.update_layout(
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    margin=dict(l=40, r=10, t=30, b=40),
    width=500, height=500,
    legend=dict(bgcolor='#1e1e1e', font=dict(color='white', size=9),
                bordercolor='#444'),
    xaxis=dict(showgrid=False, color='#666', title='X (m RD)'),
    yaxis=dict(showgrid=False, color='#666', title='Y (m RD)',
               scaleanchor='x'),
)


# ── click handler ─────────────────────────────────────────────────────────────
def _on_click(trace, points, selector):
    if not points.point_inds:
        return
    pt_idx  = points.point_inds[0]
    row_idx = int(trace.customdata[pt_idx])
    row     = inv.loc[row_idx]
    lbl     = int(row['label'])
    name    = LABEL_NAMES.get(lbl, f'Label {lbl}')
    info_html.value = (
        f'<span style="color:#ccc;font-size:12px">'f'<b>#{int(row["cluster_idx"])}  {name}</b>  '
        f'{int(row["n_raw_pts"]):,} pts  ·  {float(row["area_m2"]):.2f} m²'
        f'</span>'
    )
    detail_img.value = render_detail(row)


# ── build / update map for a tile ─────────────────────────────────────────────
def _build_map(tilecode):
    with fig_map.batch_update():
        # clear old traces and images
        fig_map.data   = []
        fig_map.layout.images = []

        b64, ext = _get_tile_bg(tilecode)
        x0, x1, y0, y1 = ext

        fig_map.update_layout(
            title=dict(text=tilecode, font=dict(color='white', size=11)),
            xaxis_range=[x0, x1],
            yaxis_range=[y0, y1],
            images=[dict(
                source=b64,
                xref='x', yref='y',
                x=x0,   y=y1,        # plotly: x=left, y=top
                sizex=x1 - x0,
                sizey=y1 - y0,
                sizing='stretch',
                layer='below',
                opacity=1.0,
            )],
        )

        tile_inv = inv[inv['tilecode'] == tilecode]
        for lbl_code, grp in tile_inv.groupby('label'):
            color = DOT_COLORS.get(lbl_code, DOT_DEFAULT)
            name  = LABEL_NAMES.get(lbl_code, f'Label {lbl_code}')
            hover = [
                f"#{r['cluster_idx']}  {name}<br>"
                f"{int(r['n_raw_pts']):,} pts  ·  {float(r['area_m2']):.1f} m²"
                for _, r in grp.iterrows()
            ]
            trace = go.Scatter(
                x=grp['centroid_x'].tolist(),
                y=grp['centroid_y'].tolist(),
                mode='markers',
                name=name,
                customdata=grp.index.tolist(),
                hovertemplate='%{text}<extra></extra>',
                text=hover,
                marker=dict(
                    color=color, size=11,
                    line=dict(color='#111111', width=1),
                ),
            )
            fig_map.add_trace(trace)

        # attach click handler to every trace
        for trace in fig_map.data:
            trace.on_click(_on_click)


def _on_tile_change(change):
    if change['name'] == 'value':
        detail_img.value = b''
        info_html.value  = '<span style="color:#666;font-size:12px">loading…</span>'
        _build_map(change['new'])
        info_html.value  = '<span style="color:#666;font-size:12px">click a cluster dot on the map</span>'

tile_dd.observe(_on_tile_change)

# ── layout ────────────────────────────────────────────────────────────────────
right = widgets.VBox(
    [info_html, detail_img],
    layout=widgets.Layout(justify_content='flex-start'),
)
ui = widgets.VBox([
    tile_dd,
    widgets.HBox([fig_map, right], layout=widgets.Layout(gap='20px')),
])

display(ui)
_build_map(TILECODES[0])

### Background colour legend
| Colour | Label |
|---|---|
| 🔴 Dark red | Road |
| 🔵 Blue | Building |
| ⚫ Mid-grey | Ground |
| 🟢 Green | Tree |
| 🟠 Orange | Car |
| 🟡 Yellow | Street light |